# Matched decoder relation memory adaptation pilot

R93 changed candidate lists but no top captions and produced an unsupported chair candidate under target swap. The next step is controlled adaptation, not more annotation or gate amplification.

Fixed protocol: 512 action-bearing COCO train/restval images, seed42, excluding all100 inspected selected-pool filenames. Both arms start from the same historical E3 V1 checkpoint. Two epochs each, batch16 images, first-five available captions per image, AdamW lr1e-5/weight_decay.01, clipnorm1. Only decoder cross-attention and norm2 train; encoder and other decoder parameters freeze. Caption CE only. Same cohort/order/parameter names and seed; different memory shapes can produce different dropout masks. No early stopping or evaluation-based tuning.

Training uses the original full V1 cache (instance_pair_grounded_relation_v1), with heuristic relations rather than human ground truth or the newest selector's predictions. Only holding/riding/carrying/wearing slots are active in both arms. The selected inference-only diagnostic cache is rejected for training. This tests decoder routing adaptation, not better prompt generation.

Both arms are evaluated on the same inspected12 development images using selected relations, on/off/target-swap. These images are disjoint from the512 training images, but this is not an untouched heldout scientific test. Report supported content and relation errors as well as sensitivity. No quality or novelty claim is established by this pilot.

The source caption checkpoint may already have seen these COCO training images during baseline training. Only the new adaptation stage excludes them; exclusion is not an independent heldout test.

16 local tests PASS: cached encoder/grounding, direct-memory checks, deterministic disjoint cohort selection, unchanged source masks, parameter scope and a finite synthetic optimizer step. The real512-image pilot has not run locally. Syntax checked.

Import DECODER_RELATION_MEMORY_ADAPTATION_512_FULL.ipynb. It is standalone. Fill SELECTED with the same mounted JSON path as the previous run. Checkpoint, prompt cache, full training grounded cache, visual cache and COCO_JSON have the previously supplied paths; attach those inputs if absent. COCO_JSON supplies captions; no COCO image files are read. No full visual-cache staging. Enable Internet and GPU; one GPU is used sequentially.

Code pinned to ef772a1. Return decoder_relation_memory_adaptation_512_review.zip (JSON evidence only). Both legacy/direct adapted_epoch_2.pth checkpoints remain separately in saved Kaggle output, excluded from the review ZIP.


In [ ]:
print("Matched decoder adaptation pilot started.", flush=True)
import json, os, shutil, subprocess, sys
from pathlib import Path

# Chi sua SELECTED thanh duong dan JSON da dung trong lan chay truoc.
SELECTED = Path("/kaggle/input/datasets/ducanh2403/CHANGE_ME/selected_relations_full100.json")
CHECKPOINT = Path("/kaggle/input/notebooks/ducanh2403/e3-checkpoint-store/e3_checkpoint_store/model_h1_2_crossattn_epoch_3.pth")
PROMPT_CACHE = Path("/kaggle/input/datasets/ducanh2403/prompt-conditioned-qformer-32q-2l/relation_prompts_v2_1_full/prompt_clip_tokens_relation_v2_1.pt")
TRAIN_GROUNDED_CACHE = Path("/kaggle/input/notebooks/ducanh2403/build-grounded-relation-v1/grounded_relation_v1_full/grounded_relation_cache.pt")
COCO_JSON = Path("/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json")
VISUAL_CACHE = Path("/kaggle/input/datasets/ducanh2403/visual-cache")
CLIP_MODEL = "openai/clip-vit-base-patch16"
REPO = Path("/kaggle/working/Image_Captioning")
COMMIT = "ef772a1"
CACHE = Path("/kaggle/working/decoder_adaptation_eval_relations.pt")
OUTPUT = Path("/kaggle/working/decoder_relation_memory_adaptation_512")
SAMPLE_SIZE, BEAM_SIZE, CANDIDATE_COUNT = 12, 5, 3

assert "CHANGE_ME" not in str(SELECTED), "Dien SELECTED bang duong dan JSON da chay thanh cong o notebook truoc."
for path in (SELECTED, CHECKPOINT, PROMPT_CACHE, TRAIN_GROUNDED_CACHE, COCO_JSON):
    assert path.is_file(), f"Missing file: {path}"
assert VISUAL_CACHE.is_dir(), f"Missing directory: {VISUAL_CACHE}"
assert not CACHE.exists() and not OUTPUT.exists(), "Neu chay lai, doi CACHE va OUTPUT sang ten moi de giu ket qua cu."
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
os.environ["PYTHONUNBUFFERED"] = "1"

def run(args, cwd=None):
    args = list(map(str, args))
    print("\nRunning:", " ".join(args), flush=True)
    subprocess.run(args, cwd=cwd, check=True)

print("Inputs verified.", flush=True)
for name, path in (("SELECTED", SELECTED), ("CHECKPOINT", CHECKPOINT), ("PROMPT_CACHE", PROMPT_CACHE), ("VISUAL_CACHE", VISUAL_CACHE)):
    print(name, "=", path, flush=True)

if not REPO.exists():
    run(["git", "clone", "https://github.com/Supzxjee/Image_Captioning.git", REPO])
assert (REPO / ".git").is_dir(), f"Not a Git checkout: {REPO}"
run(["git", "fetch", "--depth", "30", "origin", "main"], REPO)
run(["git", "checkout", "--detach", COMMIT], REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], REPO)

import torch
assert torch.cuda.is_available(), "Bat GPU trong Kaggle Settings."
print("PyTorch:", torch.__version__, "| CUDA devices:", torch.cuda.device_count(), flush=True)
for index in range(torch.cuda.device_count()):
    print("GPU", index, torch.cuda.get_device_name(index), flush=True)

run([sys.executable, "-u", "build_selected_grounded_cache.py", "--selected", SELECTED, "--output", CACHE, "--clip-model-path", CLIP_MODEL], REPO)
run([sys.executable, "-u", "train_decoder_relation_memory_pilot.py", "--checkpoint", CHECKPOINT, "--dataset-json", COCO_JSON, "--prompt-cache", PROMPT_CACHE, "--training-grounded-cache", TRAIN_GROUNDED_CACHE, "--selected", SELECTED, "--evaluation-grounded-cache", CACHE, "--visual-cache", VISUAL_CACHE, "--output", OUTPUT], REPO)
print("Download review:", str(OUTPUT) + "_review.zip", flush=True)
print("Checkpoints kept in:", OUTPUT / "legacy", "and", OUTPUT / "direct", flush=True)
